In [3]:
import requests
import pandas as pd

# --- EXTRACT --- #

API_KEY = '511b242bcf44fc36bf73ee7d37a2fad5' # Replace with your actual OpenWeatherMap API key
BASE_URL = 'http://api.openweathermap.org/data/2.5/weather'
CITIES = ['Mumbai', 'Delhi', 'Bangalore', 'Chennai', 'Hyderabad', 'Kolkata', 'Pune', 'Jaipur']

def fetch_weather(city, API_KEY):
    """
    Fetch current weather data for a given city.
    Returns a dictionary with weather metrics, or None on failure.
    """
    params = {
        'q':     city,
        'appid': API_KEY,
        'units': 'metric'
    }
    try:
        response = requests.get(BASE_URL, params=params, timeout=10)
        if response.status_code == 200:
            data = response.json()
            return {
                'city':        city,
                'temperature': round(data['main']['temp'], 1),
                'feels_like':  round(data['main']['feels_like'], 1),
                'humidity':    data['main']['humidity'],
                'pressure':    data['main']['pressure'],
                'wind_speed':  data['wind']['speed'],
                'condition':   data['weather'][0]['description'].title(),
                'visibility':  data.get('visibility', 0) // 1000
            }
        else:
            print(f'  ERROR {response.status_code} for {city}: {response.json().get("message","unknown error")}')
            return None

    except requests.exceptions.ConnectionError:
        print(f'  CONNECTION ERROR for {city} — check internet connection')
        return None
    except requests.exceptions.Timeout:
        print(f'  TIMEOUT for {city} — API did not respond in 10 seconds')
        return None


print('Calling Weather API...')
weather_records = []
for city in CITIES:
    print(f'  Fetching: {city}...', end='')
    record = fetch_weather(city, API_KEY)
    if record:
        weather_records.append(record)
        print(f' {record["temperature"]}°C, {record["condition"]}')
    else:
        print(' FAILED')

print(f'\nSuccessfully fetched: {len(weather_records)}/{len(CITIES)} cities')

# --- TRANSFORM --- #

# Create DataFrame from API results
weather_df = pd.DataFrame(weather_records)

print('\nWeather DataFrame created and data cleaned (structured):')
print(weather_df.head())
print(f'Shape: {weather_df.shape}')
print(f'Missing values: {weather_df.isnull().sum().sum()}')

# --- LOAD --- #

# Save the DataFrame to a CSV file
output_filename = 'weather_data_etl_output.csv'
weather_df.to_csv(output_filename, index=False)

print(f'\nWeather data saved to: {output_filename}')
print('\nWeather Data ETL Pipeline: COMPLETE')
print('  EXTRACT   → OpenWeatherMap API called')
print('  TRANSFORM → JSON parsed, DataFrame built, units converted')
print(f'  LOAD      → {output_filename} saved')

Calling Weather API...
  Fetching: Mumbai... 31.0°C, Haze
  Fetching: Delhi... 31.1°C, Overcast Clouds
  Fetching: Bangalore... 24.1°C, Thunderstorm
  Fetching: Chennai... 32°C, Few Clouds
  Fetching: Hyderabad... 35.2°C, Haze
  Fetching: Kolkata... 27.0°C, Haze
  Fetching: Pune... 29.7°C, Clear Sky
  Fetching: Jaipur... 38.6°C, Haze

Successfully fetched: 8/8 cities

Weather DataFrame created and data cleaned (structured):
        city  temperature  feels_like  humidity  pressure  wind_speed  \
0     Mumbai         31.0        37.6        70      1009        5.14   
1      Delhi         31.1        31.8        45       999        8.23   
2  Bangalore         24.1        24.3        67      1013        3.09   
3    Chennai         32.0        39.0        77      1009        7.72   
4  Hyderabad         35.2        38.0        41      1006        3.60   

         condition  visibility  
0             Haze           4  
1  Overcast Clouds           4  
2     Thunderstorm           6  
3

In [4]:
import pandas as pd

# --- CONFIGURATION --- #
excel_file_path = 'employee_salary_data.xlsx' # Make sure this Excel file is uploaded or accessible
output_csv_file = 'cleaned_employee_salary.csv'

# --- EXTRACT --- #
print('--- Starting Employee Salary ETL Project ---')

try:
    # Attempt to load data from Excel
    df_raw = pd.read_excel(excel_file_path)
    print(f'Successfully loaded data from {excel_file_path}. Shape: {df_raw.shape}')
except FileNotFoundError:
    print(f'Warning: {excel_file_path} not found. Creating dummy data for demonstration.')
    # Create a dummy DataFrame if the Excel file is not found
    data = {
        'Employee ID': [101, 102, 103, 104, 105, 101, 106, 107, 108],
        'Name': ['Alice Smith', 'Bob Johnson', 'Charlie Brown', 'Diana Prince', 'Eve Adams', 'Alice Smith', 'Frank White', 'Grace Black', 'Harry Green'],
        'Department': ['HR', 'IT', 'Finance', 'Marketing', 'HR', 'HR', 'IT', 'Finance', 'Marketing'],
        'Monthly Salary': [5000, 6000, 7500, 5500, 5200, 5000, 6200, 7800, 5800]
    }
    df_raw = pd.DataFrame(data)
    print(f'Dummy DataFrame created. Shape: {df_raw.shape}')

df = df_raw.copy() # Create a working copy

# --- TRANSFORM --- #
print('\n--- Data Transformation ---')

# 1. Remove duplicate records
print(f'Before dropping duplicates: {len(df)} rows')
duplicates_before = df.duplicated().sum()
df.drop_duplicates(inplace=True)
print(f'Removed {duplicates_before} duplicate rows. After dropping duplicates: {len(df)} rows')

# 2. Calculate Yearly Salary
if 'Monthly Salary' in df.columns:
    df['Yearly Salary'] = df['Monthly Salary'] * 12
    print('Calculated "Yearly Salary" based on "Monthly Salary".')
elif 'salary' in df.columns:
    df['Yearly Salary'] = df['salary'] * 12
    print('Calculated "Yearly Salary" based on "salary".')
else:
    print('Could not find "Monthly Salary" or "salary" column to calculate yearly salary.')

print('\nTransformed Data (first 5 rows):')
print(df.head())
print(f'Final DataFrame shape after transformations: {df.shape}')
print(f'Missing values after transformations: {df.isnull().sum().sum()}')

# --- LOAD --- #
print('\n--- Data Loading ---')
df.to_csv(output_csv_file, index=False)
print(f'Cleaned employee salary data saved to: {output_csv_file}')

print('\n--- Employee Salary ETL Project: COMPLETE ---')
print('  EXTRACT   → Data loaded from Excel (or dummy data generated)')
print('  TRANSFORM → Duplicates removed, Yearly Salary calculated')
print(f'  LOAD      → {output_csv_file} saved')

--- Starting Employee Salary ETL Project ---
Dummy DataFrame created. Shape: (9, 4)

--- Data Transformation ---
Before dropping duplicates: 9 rows
Removed 1 duplicate rows. After dropping duplicates: 8 rows
Calculated "Yearly Salary" based on "Monthly Salary".

Transformed Data (first 5 rows):
   Employee ID           Name Department  Monthly Salary  Yearly Salary
0          101    Alice Smith         HR            5000          60000
1          102    Bob Johnson         IT            6000          72000
2          103  Charlie Brown    Finance            7500          90000
3          104   Diana Prince  Marketing            5500          66000
4          105      Eve Adams         HR            5200          62400
Final DataFrame shape after transformations: (8, 5)
Missing values after transformations: 0

--- Data Loading ---
Cleaned employee salary data saved to: cleaned_employee_salary.csv

--- Employee Salary ETL Project: COMPLETE ---
  EXTRACT   → Data loaded from Excel (or dum